# BGE-M3 Hybrid Retrieval Evaluation on BEIR

BGE-M3 is a single model that produces three representations in one forward pass:
- **Dense**: 1024-dim `[CLS]` embedding
- **Sparse**: SPLADE-style vocabulary-weighted vector
- **Multi-vector**: ColBERT-style token embeddings (not used here)

This notebook evaluates retrieval modes on a BEIR dataset and compares them:
1. Hybrid
2. HyDE (optional)
3. Re-Rank (optional)
4. All of the above


## 1. Colab setup

In [ ]:
USE_COLAB = False   # set True when running in Google Colab

if USE_COLAB:
    !git clone -b surya/hyde-changes https://github.com/codingwithsurya/beir.git
    %cd beir
    !pip install -e .
    !pip install -U sentence-transformers datasets pytrec-eval-terrier huggingface_hub FlagEmbedding


## 2. Imports

In [ ]:
from __future__ import annotations

import logging
import os
import pathlib
import random

import numpy as np
import torch
from FlagEmbedding import BGEM3FlagModel
from tqdm.auto import tqdm

from beir import LoggingHandler, util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.models.hyde import HyDE, HyDEPromptBuilder

logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

## 3. Configuration

Change `DATASET` to any BEIR dataset name. Smaller datasets to try first:
- `nfcorpus` (~3K docs, fast)
- `scifact` (~5K docs)
- `fiqa` (~57K docs)
- `arguana` (~8.6K docs)

`DENSE_WEIGHT` and `SPARSE_WEIGHT` control hybrid fusion (BGE-M3 paper recommends `1.0` / `0.3`).

Set `USE_HYDE = True` to enable HyDE query expansion
Set `USE_RERANKER = True` to enable Reranking

In [ ]:
DATASET = "scifact"
BATCH_SIZE = 32
TOP_K = 100
DENSE_WEIGHT = 1.0
SPARSE_WEIGHT = 0.3
K_VALUES = [1, 3, 5, 10, 100]

# HyDE settings
USE_HYDE = False                          # set True to enable HyDE query expansion
HYDE_LOCAL_MODEL = "google/flan-t5-base"  # runs on CPU/MPS, no API key needed
                                          # use "google/flan-t5-large" for better quality (needs GPU)
HYDE_N_HYPOTHESES = 5                     # hypothetical documents generated per query
HYDE_MAX_NEW_TOKENS = 128                 # flan-t5-base degrades beyond ~128 tokens
HYDE_INCLUDE_ORIGINAL_QUERY = True        # average hypotheses with the original query embedding

# Reranking settings
USE_RERANKER = True                      # set True to rerank hybrid results with BGE-reranker-v2-m3
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
RERANKER_BATCH_SIZE = 4   # keep small on MPS/CPU to avoid OOM; use 32+ on CUDA
RERANKER_TOP_K = 100                      # number of hybrid candidates to rerank

# Resolve paths based on environment
BASE_DIR  = "/content" if USE_COLAB else str(pathlib.Path(".").absolute())
DATA_DIR  = os.path.join(BASE_DIR, "datasets")
CACHE_DIR = os.path.join(BASE_DIR, "hyde_cache")

## 4. Download and load dataset

In [ ]:
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET}.zip"
data_path = util.download_and_unzip(url, DATA_DIR)

corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

print(f"Corpus size: {len(corpus):,}")
print(f"Queries:     {len(queries):,}")
print(f"Qrels:       {len(qrels):,}")

## 5. Load BGE-M3

In [ ]:
model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=device == "cuda",
    device=device,
)
print("Model loaded.")

## 6. Encode corpus

Corpus encoding is the same regardless of whether HyDE is used — documents are always encoded normally.

In [ ]:
doc_ids = list(corpus.keys())
doc_texts = [f"{corpus[d].get('title', '')} {corpus[d].get('text', '')}".strip() for d in doc_ids]

print(f"Encoding {len(doc_texts):,} documents...")
corpus_output = model.encode(
    doc_texts,
    batch_size=BATCH_SIZE,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)

corpus_dense  = corpus_output["dense_vecs"]       # (num_docs, 1024)
corpus_sparse = corpus_output["lexical_weights"]   # list of {token_id: weight}

print(f"Done. Dense shape: {corpus_dense.shape}")

## 7. Encode queries (with optional HyDE expansion)

**Without HyDE:** each query is encoded directly by BGE-M3.

**With HyDE:** an LLM generates `HYDE_N_HYPOTHESES` hypothetical answer passages per query. Each hypothesis is encoded as a *document* (same encoder as corpus), then averaged with the original query embedding. This gives a query vector that lives in document space, typically improving recall.

The HyDE wrapper caches generated hypotheses to a JSONL file so you don't re-call the API on re-runs.

In [ ]:
query_ids   = list(queries.keys())
query_texts = [queries[q] for q in query_ids]

if USE_HYDE:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    print(f"HyDE enabled — loading local generator: {HYDE_LOCAL_MODEL}")

    _gen_device = torch.device("cuda" if device == "cuda" else "cpu")
    _tokenizer  = AutoTokenizer.from_pretrained(HYDE_LOCAL_MODEL)
    _gen_model  = AutoModelForSeq2SeqLM.from_pretrained(HYDE_LOCAL_MODEL).to(_gen_device)
    _gen_model.eval()

    class LocalGenerator:
        def generate(self, prompt: str) -> list[str]:
            # Batch all hypotheses in one forward pass instead of one at a time
            inputs = _tokenizer(
                [prompt] * HYDE_N_HYPOTHESES,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True,
            ).to(_gen_device)
            with torch.no_grad():
                outputs = _gen_model.generate(
                    **inputs,
                    max_new_tokens=HYDE_MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.7,
                )
            return _tokenizer.batch_decode(outputs, skip_special_tokens=True)

    class BGEM3Adapter:
        def encode_queries(self, texts, batch_size=32, **kwargs):
            out = model.encode(texts, batch_size=batch_size, return_dense=True,
                               return_sparse=False, return_colbert_vecs=False)
            return torch.from_numpy(out["dense_vecs"])

        def encode_corpus(self, texts, batch_size=32, **kwargs):
            if texts and isinstance(texts[0], dict):
                texts = [f"{t.get('title', '')} {t.get('text', '')}".strip() for t in texts]
            out = model.encode(texts, batch_size=batch_size, return_dense=True,
                               return_sparse=False, return_colbert_vecs=False)
            return torch.from_numpy(out["dense_vecs"])

    hyde = HyDE(
        base_model=BGEM3Adapter(),
        generator=LocalGenerator(),
        dataset=DATASET,
        cache_path=os.path.join(CACHE_DIR, f"{DATASET}.{HYDE_LOCAL_MODEL.replace('/', '_')}.jsonl"),
        include_original_query=HYDE_INCLUDE_ORIGINAL_QUERY,
        hypothesis_encoder="corpus",
    )

    query_dense_np = hyde.encode_queries(query_texts, batch_size=BATCH_SIZE)
    if isinstance(query_dense_np, torch.Tensor):
        query_dense_np = query_dense_np.cpu().numpy()
    query_dense = query_dense_np

    print("Encoding query sparse representations (no HyDE for sparse)...")
    query_sparse_out = model.encode(
        query_texts, batch_size=BATCH_SIZE,
        return_dense=False, return_sparse=True, return_colbert_vecs=False,
    )
    query_sparse = query_sparse_out["lexical_weights"]

else:
    print(f"Encoding {len(query_texts):,} queries...")
    query_output = model.encode(
        query_texts,
        batch_size=BATCH_SIZE,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    query_dense  = query_output["dense_vecs"]
    query_sparse = query_output["lexical_weights"]

print(f"Query dense shape: {query_dense.shape}")
print("Query encoding complete.")

## 8. Score functions

In [ ]:
def dense_scores(query_vecs: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    return query_vecs @ doc_vecs.T  # (num_queries, num_docs)


def sparse_score(query_weights: dict, doc_weights: dict) -> float:
    return sum(w * doc_weights[t] for t, w in query_weights.items() if t in doc_weights)


def min_max_normalize(matrix: np.ndarray) -> np.ndarray:
    mins = matrix.min(axis=1, keepdims=True)
    maxs = matrix.max(axis=1, keepdims=True)
    return (matrix - mins) / (maxs - mins + 1e-9)


def build_results(score_matrix: np.ndarray, query_ids: list, doc_ids: list, top_k: int) -> dict:
    results = {}
    for i, qid in enumerate(query_ids):
        scores = score_matrix[i]
        top_idx = np.argpartition(scores, -top_k)[-top_k:]
        results[qid] = {doc_ids[j]: float(scores[j]) for j in top_idx}
    return results

## 9. Dense retrieval

In [ ]:
print("Running dense retrieval...")
dense_score_matrix = dense_scores(query_dense, corpus_dense)
dense_results = build_results(dense_score_matrix, query_ids, doc_ids, TOP_K)
print("Done.")

## 10. Sparse retrieval

Computing sparse dot products for every (query, doc) pair. For large corpora you would use an inverted index instead — this brute-force loop is only practical for small datasets like `nfcorpus`.

In [ ]:
print("Running sparse retrieval...")
sparse_score_matrix = np.zeros((len(query_ids), len(doc_ids)), dtype=np.float32)

for i, qw in enumerate(tqdm(query_sparse, desc="Sparse scoring")):
    for j, dw in enumerate(corpus_sparse):
        sparse_score_matrix[i, j] = sparse_score(qw, dw)

sparse_results = build_results(sparse_score_matrix, query_ids, doc_ids, TOP_K)
print("Done.")

## 11. Hybrid retrieval (weighted sum fusion)

Both score matrices are min-max normalized per query before fusion so their scales are comparable.

In [ ]:
print("Running hybrid retrieval...")
hybrid_score_matrix = (
    DENSE_WEIGHT  * min_max_normalize(dense_score_matrix)
    + SPARSE_WEIGHT * min_max_normalize(sparse_score_matrix)
)
hybrid_results = build_results(hybrid_score_matrix, query_ids, doc_ids, TOP_K)
print("Done.")

## 12. Reranking (optional)

`BAAI/bge-reranker-v2-m3` is a cross-encoder from the same model family as BGE-M3. It takes the top-`RERANKER_TOP_K` hybrid candidates and re-scores each `(query, document)` pair jointly, which is much more accurate than the bi-encoder dot product used in first-stage retrieval.

Set `USE_RERANKER = True` in the config cell to enable this step.

In [ ]:
if USE_RERANKER:
    from beir.reranking.models import CrossEncoder
    from beir.reranking import Rerank

    print(f"Loading reranker: {RERANKER_MODEL}")
    reranker = Rerank(
        CrossEncoder(RERANKER_MODEL),
        batch_size=RERANKER_BATCH_SIZE,
    )

    print(f"Reranking top-{RERANKER_TOP_K} candidates per query...")
    reranked_results = reranker.rerank(corpus, queries, hybrid_results, top_k=RERANKER_TOP_K)
    print("Reranking complete.")
else:
    reranked_results = None
    print("Reranker disabled — set USE_RERANKER = True in config to enable.")

## 12. Evaluate all modes

In [ ]:
evaluator = EvaluateRetrieval()
hyde_tag = " + HyDE" if USE_HYDE else ""

print("=" * 60)
print(f"Dataset: {DATASET}  |  HyDE: {USE_HYDE}  |  Reranker: {USE_RERANKER}")
print("=" * 60)

runs = [(f"Hybrid{hyde_tag}", hybrid_results)]
if reranked_results:
    runs.append((f"Hybrid{hyde_tag} + Reranker", reranked_results))

for name, results in runs:
    ndcg, map_, recall, precision = evaluator.evaluate(qrels, results, K_VALUES)
    print(f"\n--- {name} ---")
    print(f"  NDCG@10:    {ndcg['NDCG@10']:.4f}")
    print(f"  Recall@100: {recall['Recall@100']:.4f}")
    print(f"  MAP@10:     {map_['MAP@10']:.4f}")

## 13. Inspect top-k results for a random query

In [ ]:
query_id = random.choice(query_ids)
print(f"Query: {queries[query_id]}\n")

hyde_tag = " + HyDE" if USE_HYDE else ""
runs = [(f"Hybrid{hyde_tag}", hybrid_results)]
if reranked_results:
    runs.append((f"Hybrid{hyde_tag} + Reranker", reranked_results))

for name, results in runs:
    print(f"--- Top 5 ({name}) ---")
    ranked = sorted(results[query_id].items(), key=lambda x: x[1], reverse=True)[:5]
    for rank, (doc_id, score) in enumerate(ranked, 1):
        title = corpus[doc_id].get("title", "")[:80]
        relevant = "*" if doc_id in qrels.get(query_id, {}) else " "
        print(f"  {relevant} {rank}. [{score:.4f}] {title}")
    print()